# TRISTAR Vehicles Batch Ingestion

Load vehicle reference data into the Bronze layer.

## 1. Configuration

Define Unity Catalog and Volume paths for batch ingestion.

In [0]:
catalog = "dbr_dev_ua5816bd"
schema_bronze = "team_tristar_bronze"
volume = "raw_data"

base_path = f"/Volumes/{catalog}/{schema_bronze}/{volume}"

batch_path = f"{base_path}/batch"
archive_path = f"{batch_path}/archive"


## 2. Vehicles Source

In [0]:
import requests
from datetime import datetime

vehicles_url = (
    "https://files.cloudgdansk.pl/d/otwarte-dane/ztm/"
    "baza-pojazdow.json?v=2"
)

vehicles_response = requests.get(vehicles_url, timeout=30)
vehicles_response.raise_for_status()

vehicles_payload = vehicles_response.json()

vehicles_source_timestamp = vehicles_payload["metadata"]["sourceDate"]

vehicles_source_date = datetime.fromisoformat(
    vehicles_source_timestamp.replace("Z", "+00:00")
).strftime("%Y-%m-%d")

In [0]:
vehicles_archive_path = f"{archive_path}/vehicles"
dbutils.fs.mkdirs(vehicles_archive_path)

vehicles_file_name = f"vehicles_{vehicles_source_date}.json"
vehicles_file = f"{vehicles_archive_path}/{vehicles_file_name}"

existing_vehicle_files = {
    file.name
    for file in dbutils.fs.ls(vehicles_archive_path)
}

new_vehicles_snapshot = (
    vehicles_file_name not in existing_vehicle_files
)

if new_vehicles_snapshot:
    with open(vehicles_file, "w", encoding="utf-8") as file:
        file.write(vehicles_response.text)

In [0]:
vehicles_table = f"{catalog}.{schema_bronze}.vehicles"

load_vehicles = True

if spark.catalog.tableExists(vehicles_table):
    loaded_date = (
        spark.table(vehicles_table)
        .first()["source_update_date"]
    )

    if str(loaded_date) == vehicles_source_date:
        load_vehicles = False

In [0]:
from pyspark.sql.functions import lit, current_timestamp

if load_vehicles:

    vehicles_raw_df = (
        spark.read
            .option("multiline", "true")
            .json(vehicles_file)
    )

    vehicles_df = (
        vehicles_raw_df
        .selectExpr("explode(results) as vehicle")
        .select("vehicle.*")
    )

    vehicles_bronze_df = (
        vehicles_df
        .withColumn(
            "source",
            lit("tristar_vehicles/baza-pojazdow.json")
        )
        .withColumn(
            "source_update_date",
            lit(vehicles_source_date).cast("date")
        )
        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )
    )

    (
        vehicles_bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(vehicles_table)
    )